# Notebook 1: PlanetScope preprocessing and EDA

This notebook creates the model-ready Gaza dataset from the downloaded PlanetScope raster. The image tensor contains only matching pre-event and post-event PlanetScope bands; footprint geometry and labels remain in the row-aligned parquet table.

The target is the 3 May 2024 UNOSAT assessment. Because the Planet post composite covers June, the 6 July assessment is used only as a false-negative guard: a footprint that is intact in May but damaged in July is removed. Both label files come from notebook 0 and use UNOSAT damage codes 1-3 as the positive class.

Outputs:

- `data/processed/Gaza_20240503_320a78e597.npz`
- `data/processed/Gaza_20240503_320a78e597.parquet`

The NPZ contains eight 3 m Planet bands (pre/post B, G, R and NIR) in `(N, C, H, W)` order. The parquet contains the same buildings in exactly the same row order. Later modeling notebooks read this saved pair.


## 1. Colab setup

Mount Drive and install the geospatial readers used by the shared pipeline. The install cell is safe to rerun in a fresh runtime.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q rasterio geopandas pyarrow


### 1a. Project paths

`PROJECT_DIR` identifies the Drive folder containing this notebook, `pipeline.py`, `data/rasters/` and `data/processed/`. The default matches the repository layout. Labeled footprints come from shared notebook 0 in the project's `shared/data/labeled_footprints/` folder.


In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/War-Damage-Detection')
PROJECT_DIR = PROJECT_ROOT / 'planet'
LABEL_DIR = PROJECT_ROOT / 'shared' / 'data' / 'labeled_footprints'

if not (PROJECT_DIR / 'pipeline.py').exists():
    raise FileNotFoundError(
        f'{PROJECT_DIR / "pipeline.py"} not found. Adjust PROJECT_DIR above.')

os.environ['PLANET_DAMAGE_BASE'] = str(PROJECT_DIR)
os.environ['PLANET_LABEL_DIR'] = str(LABEL_DIR)
os.environ['PLANET_TEMP_DIR'] = '/content/planet_damage_tmp'
for path in (PROJECT_ROOT, PROJECT_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from planet.pipeline import (
    BASE_DIR, LABEL_DIR, RASTER_DIR, DATASET_DIR, TEMP_DIR, PREP,
    MAY_DATE, JULY_DATE, DATASET_STEM, PLANET_CHANNELS,
    label_path, planet_raster_path, dataset_paths, preflight,
    validate_raster, extract_dataset,
    inspect_dataset, read_saved_patches, preprocessing_fingerprint,
)

legacy_moves = [
    (PROJECT_ROOT / 'Data' / 'own_footprints', LABEL_DIR),
    (PROJECT_ROOT / 'Planet Data (Nils)' / 'rasters', RASTER_DIR),
    (PROJECT_ROOT / 'Planet Data (Nils)' / 'datasets', DATASET_DIR),
]
for old_path, new_path in legacy_moves:
    if old_path.is_dir() and not new_path.is_dir():
        raise FileNotFoundError(
            f'Move the existing Drive folder {old_path} to {new_path} ' 
            'before running preprocessing.')

print('project folder: ', BASE_DIR)
print('label folder:   ', LABEL_DIR)
print('raster folder:  ', RASTER_DIR)
print('dataset folder: ', DATASET_DIR)
print('temporary files:', TEMP_DIR)
print('dataset stem:   ', DATASET_STEM)
print('fingerprint:    ', preprocessing_fingerprint())


### 1b. Preflight

Confirm both label dates and the Planet TIFF are present before doing expensive work. A complete existing dataset is validated and skipped unless `FORCE=True` is set later.


In [ ]:
inventory = preflight()
display(inventory)

required = inventory[inventory['kind'].isin([
    f'labels {MAY_DATE}', f'labels {JULY_DATE}', 'Planet raster'])]
if not required['exists'].all():
    missing = required.loc[~required['exists'], 'path'].tolist()
    raise FileNotFoundError(f'Missing required inputs: {missing}')

print('Preprocessing configuration:')
for key, value in PREP.items():
    print(f'  {key}: {value}')
print(f"\nPatch footprint: {PREP['patch_size']} px × {PREP['scale_m']} m "
      f"= {PREP['patch_size'] * PREP['scale_m']} m per side")


## 2. May labels with the July exclusion rule

The two CSVs must contain the same footprint IDs and geometries. May supplies `y`; July is never used as a feature or replacement label. It only removes May negatives whose building was damaged by July, since that later destruction may already be visible in June imagery.


In [ ]:
may_raw = pd.read_csv(label_path(MAY_DATE), usecols=['system:index', 'class'])
july_raw = pd.read_csv(label_path(JULY_DATE), usecols=['system:index', 'class'])
transition = may_raw.merge(
    july_raw, on='system:index', validate='one_to_one',
    suffixes=(f'_{MAY_DATE}', f'_{JULY_DATE}'))

print('May to July label transition before filtering:')
display(pd.crosstab(
    transition[f'class_{MAY_DATE}'], transition[f'class_{JULY_DATE}'],
    rownames=[f'class {MAY_DATE}'], colnames=[f'class {JULY_DATE}']))

may_y = transition[f'class_{MAY_DATE}'].to_numpy(int)
july_y = transition[f'class_{JULY_DATE}'].to_numpy(int)
may_positive_july_negative = (may_y == 1) & (july_y == 0)
future_damage = (may_y == 0) & (july_y == 1)
keep = ~future_damage
label_summary = {
    'may_rows': len(transition),
    'future_damage_removed': int(future_damage.sum()),
    'may_positive_july_negative_kept': int(may_positive_july_negative.sum()),
    'filtered_rows_before_raster_checks': int(keep.sum()),
    'damaged': int(may_y[keep].sum()),
    'intact': int((may_y[keep] == 0).sum()),
    'damaged_fraction': float(may_y[keep].mean()),
}
display(pd.Series(label_summary, name='value').to_frame())
print('Applied rule: remove only the May=0, July=1 cell.')
if may_positive_july_negative.any():
    print(f'Kept {may_positive_july_negative.sum():,} May positives that are July negatives. '
          'July is an exclusion guard, not a replacement for the May target.')
print(f"Remaining prevalence: {100 * label_summary['damaged_fraction']:.2f}%")


## 3. Validate the Planet raster

The raster is expected to be a single eight-band stack at 3 m: four pre-event bands followed by four post-event bands. The band descriptions are checked so a channel-order change cannot silently corrupt the pre/post comparison.


In [ ]:
raster_info = validate_raster()
display(pd.Series(raster_info, name='value').to_frame())
print('Model channel order:')
for i, name in enumerate(PLANET_CHANNELS):
    print(f'  {i}: {name}')


## 4. Extract and save the patches

This is the expensive cell. It processes buildings in raster-block order, writes pixels to a temporary float16 memory map, rejects incomplete or nodata-contaminated patches, and finally writes the requested compressed NPZ/parquet pair. Existing complete outputs are validated and skipped.

The uncompressed tensor is roughly 3.3 GB before raster-coverage filtering. Extraction therefore requires several additional gigabytes for the temporary array and final NPZ. `FORCE=True` rebuilds both outputs; the default reuses a complete validated pair.


In [ ]:
FORCE = False
dataset_info = extract_dataset(force=FORCE, progress_every=10_000)

summary_for_display = {k: v for k, v in dataset_info.items() if k != 'metadata'}
display(pd.Series(summary_for_display, name='value').to_frame())


## 5. Dataset integrity

Validation reads the small arrays and parquet table plus the NPZ header. It checks row counts, building IDs, channel order, tensor shape, and preprocessing fingerprint without decompressing all of `X`.


In [ ]:
info = inspect_dataset(validate_alignment=True)
npz_path, parquet_path = dataset_paths()

print(f'NPZ:     {npz_path}')
print(f'Parquet: {parquet_path}')
print(f"Rows: {info['rows']:,} | damaged: {info['damaged']:,} "
      f"({100 * info['damaged_fraction']:.2f}%)")
print(f"X: {info['X_shape']} {info['X_dtype']} | {info['npz_GB']:.2f} GB compressed")
print(f"Channels: {info['channels']}")

extraction = info['metadata']
display(pd.Series({
    key: extraction[key] for key in [
        'geometric_candidates', 'outside_raster_removed',
        'masked_removed', 'nonfinite_removed', 'saved_rows']
}, name='count').to_frame())


## 6. Planet-specific EDA

The plots below re-read only a small stratified sample directly from the raster using the exact saved patch centres. This avoids inflating the full compressed NPZ in notebook 1. Full `X` loading belongs in the modeling notebook.


In [ ]:
table = gpd.read_parquet(parquet_path)
rng = np.random.default_rng(PREP['seed'])
N_PER_CLASS = 1_000
sample_positions = []
for label in (0, 1):
    available = np.flatnonzero(table['class'].to_numpy(int) == label)
    sample_positions.extend(
        rng.choice(available, size=min(N_PER_CLASS, len(available)), replace=False))
sample_positions = np.asarray(sample_positions, dtype=int)
sample_patches, sample_table = read_saved_patches(sample_positions)
sample_y = sample_table['class'].to_numpy(int)

print(f'Read {len(sample_patches):,} patches for EDA only.')
print(pd.Series(sample_y).value_counts().sort_index().rename(index={0: 'intact', 1: 'damaged'}))


In [ ]:
# Patch-mean reflectance and post-minus-pre changes on the balanced sample.
means = sample_patches.mean(axis=(2, 3))
change = means[:, 4:8] - means[:, 0:4]
bands = ['B', 'G', 'R', 'NIR']

rows = []
for label, label_name in [(0, 'intact'), (1, 'damaged')]:
    mask = sample_y == label
    for j, band in enumerate(bands):
        rows.append({
            'class': label_name, 'band': band,
            'mean pre': means[mask, j].mean(),
            'mean post': means[mask, j + 4].mean(),
            'mean post-pre': change[mask, j].mean(),
        })
change_summary = pd.DataFrame(rows)
display(change_summary)

fig, axes = plt.subplots(1, 4, figsize=(14, 3.2))
for j, (ax, band) in enumerate(zip(axes, bands)):
    ax.hist(change[sample_y == 0, j], bins=50, density=True, alpha=.55, label='intact')
    ax.hist(change[sample_y == 1, j], bins=50, density=True, alpha=.55, label='damaged')
    ax.axvline(0, color='black', linewidth=.8, linestyle='--')
    ax.set_title(f'{band}: post − pre')
    ax.set_xlabel('surface reflectance difference')
axes[0].legend()
plt.tight_layout()
plt.show()


In [ ]:
# Example pre/post RGB pairs. Percentile scaling is shared within each pair.
rgb_pre = [PLANET_CHANNELS.index(c) for c in ('ps_pre_R', 'ps_pre_G', 'ps_pre_B')]
rgb_post = [PLANET_CHANNELS.index(c) for c in ('ps_post_R', 'ps_post_G', 'ps_post_B')]

def rgb_pair(patch):
    pre = np.moveaxis(patch[rgb_pre], 0, -1)
    post = np.moveaxis(patch[rgb_post], 0, -1)
    both = np.concatenate([pre.reshape(-1, 3), post.reshape(-1, 3)], axis=0)
    lo, hi = np.percentile(both, [2, 98], axis=0)
    scale = np.maximum(hi - lo, 1e-6)
    return np.clip((pre - lo) / scale, 0, 1), np.clip((post - lo) / scale, 0, 1)

examples = []
for label in (0, 1):
    candidates = np.flatnonzero(sample_y == label)
    examples.extend(rng.choice(candidates, size=3, replace=False))

fig, axes = plt.subplots(4, 3, figsize=(8, 10))
for col, j in enumerate(examples[:3]):
    pre, post = rgb_pair(sample_patches[j])
    axes[0, col].imshow(pre); axes[1, col].imshow(post)
    axes[0, col].set_title('intact: pre'); axes[1, col].set_title('intact: post')
for col, j in enumerate(examples[3:]):
    pre, post = rgb_pair(sample_patches[j])
    axes[2, col].imshow(pre); axes[3, col].imshow(post)
    axes[2, col].set_title('damaged: pre'); axes[3, col].set_title('damaged: post')
for ax in axes.ravel():
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle('PlanetScope 3 m patches with a shared stretch per pair', y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# Label geography. This also makes the future spatial train/val/test split visible.
plot_sample = table.sample(min(40_000, len(table)), random_state=PREP['seed'])
fig, ax = plt.subplots(figsize=(7, 10))
for label, color, name in [(0, 'tab:blue', 'intact through July'),
                           (1, 'tab:red', 'damaged by May')]:
    part = plot_sample[plot_sample['class'] == label]
    ax.scatter(part['lon'], part['lat'], s=1.5, alpha=.35, c=color, label=name)
ax.set_xlabel('longitude'); ax.set_ylabel('latitude')
ax.set_title('Saved Planet dataset labels')
ax.legend(markerscale=5)
plt.show()


## 7. Model input contract

The pretrained RGB model selects channels by name from the same eight-channel tensor. This check verifies its channel lookup without loading the full image array.


In [ ]:
channel_names = list(info['channels'])
pre_ch = [channel_names.index(c) for c in ['ps_pre_R', 'ps_pre_G', 'ps_pre_B']]
post_ch = [channel_names.index(c) for c in ['ps_post_R', 'ps_post_G', 'ps_post_B']]

assert info['X_shape'][1:] == (8, 32, 32)
assert len(table) == info['X_shape'][0]
assert pre_ch == [2, 1, 0] and post_ch == [6, 5, 4]
print('Pretrained-model RGB lookup passed.')
print('pre RGB indices: ', pre_ch)
print('post RGB indices:', post_ch)


## Done

The `data/processed/` folder now contains the fixed NPZ/parquet pair. Later Planet modeling and mapping notebooks use `inspect_dataset()` for a cheap integrity check and `load_dataset(load_images=True)` when the complete image tensor is needed.

Notebook 2 can now focus exclusively on spatial splitting, normalization using training statistics, model training, and evaluation.
